# Canadian pension surplus and LDI

Compare the existing bond hedge with alternative 15-year swap overlays. The committed tables use 20,000 common paths and the October 21, 2026 curve.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
reports = root / "reports"
display(Image(filename=str(root / "figures" / "ldi-tradeoff.png")))

## Solve the hedge decision twice
Minimize one-year surplus-at-risk on the 1 percentage point grid. Then require the 30-day cash coverage ratio to be at least one under the default stress.

In [ ]:
frontier = pd.read_csv(reports / "frontier.csv")
comparison = pd.read_csv(reports / "comparison.csv").set_index("case")
feasible = frontier[frontier["coverage"] >= 1]
selection = pd.concat([
    comparison,
    pd.DataFrame([frontier.loc[frontier["sar95"].idxmin()], feasible.loc[feasible["sar95"].idxmin()]],
                 index=["minimum SaR", "minimum SaR with cash coverage"]),
])
show = selection[["hedge_ratio", "sar95", "expected_surplus", "p_underfunded", "liquidity_buffer"]].copy()
for column in ["sar95", "expected_surplus", "liquidity_buffer"]:
    show[column] /= 1e6
display(show.rename(columns={key: f"{key} (CAD M)" for key in ["sar95", "expected_surplus", "liquidity_buffer"]}).round(4))

## Inspect risk and cash
Euler contributions sum to surplus volatility. The cash table compares monetizable assets against swap variation margin, the private capital call and benefits.

In [ ]:
display(pd.read_csv(reports / "euler.csv").pivot(index="label",columns="case",values="value").div(1e6).round(2))
display(pd.read_csv(reports / "liquidity.csv", index_col="case").drop(columns="coverage").T.div(1e6).round(2))

## Re-run with the local data cache
Follow the README download command before changing the flag.

In [ ]:
rerun = False
if rerun:
    from pension_ldi.analysis import build_study
    from pension_ldi.market import load_market, load_curve

    curve_path = root / "data" / "curve.csv"
    as_of = str(pd.read_csv(curve_path)["date"].iloc[0])
    study = build_study(load_market(root / "data" / "market.csv"), load_curve(curve_path), as_of)
    display(pd.DataFrame(study["key_rates"]))